# NER Model Training for Calendar Bot

NER на базе `bert-base-multilingual-cased` для извлечения сущностей из текстовых запросов:
- DATE (дата)
- TIME (время)
- LOC (место)
- TITLE (название события)
- USER (участники)
- URL (ссылки)

## Параметры обучения (как в описании проекта)
- Learning rate: 2e-5
- Epochs: 3
- Batch size: 16
- Validation split: 10%


## 1. Imports & Setup


Я запускала локально, поэтому у меня `FORCE_DEVICE = "cpu"`  
Это можно поменять:  

Варианты: "auto", "cpu", "mps", "cuda"
 - "auto" - автоматический выбор лучшего доступного
 - "cpu"  - принудительно CPU (медленно, но стабильно)
 - "mps"  - Apple Silicon GPU
 - "cuda" - NVIDIA GPU (Kaggle/Colab)

In [ ]:
import json
import os
import numpy as np
import torch
from pathlib import Path
from typing import Dict, List, Any

from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification,
    TrainingArguments,
    Trainer,
    DataCollatorForTokenClassification,
    EarlyStoppingCallback,
)
from seqeval.metrics import (
    classification_report,
    f1_score,
    precision_score,
    recall_score,
)
from sklearn.model_selection import train_test_split

FORCE_DEVICE = "cpu"

if FORCE_DEVICE == "auto":
    if torch.cuda.is_available():
        DEVICE = torch.device("cuda")
        print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
    elif torch.backends.mps.is_available():
        DEVICE = torch.device("mps")
        print("Using Apple Silicon MPS")
    else:
        DEVICE = torch.device("cpu")
        print("Using CPU")
elif FORCE_DEVICE == "cuda" and torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
elif FORCE_DEVICE == "mps" and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
    print("Using Apple Silicon MPS")
else:
    DEVICE = torch.device("cpu")
    print("Using CPU (forced or fallback)")

print(f"PyTorch version: {torch.__version__}")
print(f"Selected device: {DEVICE}")


/Users/milana/HSE/advanced_python_2025/python_adv_proj/venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


Using CPU (forced or fallback)
PyTorch version: 2.8.0
Selected device: cpu


## 2. Configuration


In [ ]:
PROJECT_ROOT = Path("..")
DATA_PATH = PROJECT_ROOT / "data" / "dataset.jsonl"
OUTPUT_DIR = Path("./checkpoints")
FINAL_MODEL_DIR = Path("./trained_model")

MODEL_NAME = "bert-base-multilingual-cased"

LEARNING_RATE = 2e-5
NUM_EPOCHS = 3
VALIDATION_SPLIT = 0.1
MAX_LENGTH = 128
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.1

if DEVICE.type == "cuda":
    BATCH_SIZE = 16
    GRADIENT_ACCUMULATION_STEPS = 1
    print("Config for CUDA GPU - full batch size")
elif DEVICE.type == "mps":
    BATCH_SIZE = 2
    GRADIENT_ACCUMULATION_STEPS = 8
    print("Config for Apple Silicon MPS - reduced batch size")
else:
    BATCH_SIZE = 8
    GRADIENT_ACCUMULATION_STEPS = 2
    print("Config for CPU")

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

print(f"Batch size: {BATCH_SIZE}")
print(f"Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")


Config for CPU
Batch size: 8
Gradient accumulation: 2
Effective batch size: 16


## 3. Define Label Mappings

BIO-схема тегов для 6 типов сущностей + O (outside)


In [ ]:
LABEL_LIST = [
    "O",       
    "B-DATE",  
    "I-DATE",  
    "B-TIME",
    "I-TIME",   
    "B-LOC",
    "I-LOC", 
    "B-TITLE", 
    "I-TITLE", 
    "B-USER", 
    "I-USER",  
    "B-URL",   
    "I-URL", 
]

# Маппинги label <-> id
label2id = {label: idx for idx, label in enumerate(LABEL_LIST)}
id2label = {idx: label for idx, label in enumerate(LABEL_LIST)}

NUM_LABELS = len(LABEL_LIST)
print(f"Number of labels: {NUM_LABELS}")
print(f"Labels: {LABEL_LIST}")


Number of labels: 13
Labels: ['O', 'B-DATE', 'I-DATE', 'B-TIME', 'I-TIME', 'B-LOC', 'I-LOC', 'B-TITLE', 'I-TITLE', 'B-USER', 'I-USER', 'B-URL', 'I-URL']


## 4. Load Dataset


In [ ]:
def load_jsonl(file_path: Path) -> List[Dict[str, Any]]:
    data = []
    with open(file_path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                data.append(json.loads(line))
    return data

raw_data = load_jsonl(DATA_PATH)
print(f"Loaded {len(raw_data)} examples")

print("\nПример данных:")
print(json.dumps(raw_data[0], ensure_ascii=False, indent=2))


Loaded 10000 examples

Пример данных:
{
  "id": 246547,
  "lang": "en",
  "text": "Cancel meeting about Q1 Goals, please, it was scheduled for 29 12 2025 at 10.30.",
  "tokens": [
    "Cancel",
    "meeting",
    "about",
    "Q1",
    "Goals",
    ",",
    "please",
    ",",
    "it",
    "was",
    "scheduled",
    "for",
    "29",
    "12",
    "2025",
    "at",
    "10",
    ".",
    "30",
    "."
  ],
  "ner_tags": [
    "O",
    "B-TITLE",
    "I-TITLE",
    "I-TITLE",
    "I-TITLE",
    "O",
    "O",
    "O",
    "O",
    "O",
    "O",
    "O",
    "B-DATE",
    "I-DATE",
    "I-DATE",
    "O",
    "B-TIME",
    "I-TIME",
    "I-TIME",
    "O"
  ],
  "slots": {
    "event_name": "meeting about Q1 Goals",
    "date": "29 12 2025",
    "time": "10.30"
  }
}


In [ ]:
def convert_tags_to_ids(examples: List[Dict]) -> List[Dict]:
    """Преобразование строковых тегов в числовые ID."""
    converted = []
    for ex in examples:
        tag_ids = [label2id.get(tag, 0) for tag in ex["ner_tags"]]
        converted.append({
            "id": ex["id"],
            "lang": ex["lang"],
            "tokens": ex["tokens"],
            "ner_tags": tag_ids,
            "text": ex["text"],
        })
    return converted

processed_data = convert_tags_to_ids(raw_data)
print(f"Converted {len(processed_data)} examples")
print(f"\nПример конвертированных тегов:")
print(f"Tokens: {processed_data[0]['tokens'][:10]}...")
print(f"Tags: {processed_data[0]['ner_tags'][:10]}...")


Converted 10000 examples

Пример конвертированных тегов:
Tokens: ['Cancel', 'meeting', 'about', 'Q1', 'Goals', ',', 'please', ',', 'it', 'was']...
Tags: [0, 7, 8, 8, 8, 0, 0, 0, 0, 0]...


## 5. Train/Validation Split


In [ ]:
train_data, val_data = train_test_split(
    processed_data,
    test_size=VALIDATION_SPLIT,
    random_state=SEED,
)

print(f"Train size: {len(train_data)}")
print(f"Validation size: {len(val_data)}")

train_dataset = Dataset.from_list(train_data)
val_dataset = Dataset.from_list(val_data)

dataset = DatasetDict({
    "train": train_dataset,
    "validation": val_dataset,
})

print(f"\nDataset structure:")
print(dataset)


Train size: 9000
Validation size: 1000

Dataset structure:
DatasetDict({
    train: Dataset({
        features: ['id', 'lang', 'tokens', 'ner_tags', 'text'],
        num_rows: 9000
    })
    validation: Dataset({
        features: ['id', 'lang', 'tokens', 'ner_tags', 'text'],
        num_rows: 1000
    })
})


## 6. Tokenization with Label Alignment

При использовании BERT токенизатора слова могут разбиваться на подтокены (subwords).
Нужно правильно выровнять метки для каждого подтокена.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f"Tokenizer loaded: {MODEL_NAME}")
print(f"Vocab size: {tokenizer.vocab_size}")


Tokenizer loaded: bert-base-multilingual-cased
Vocab size: 119547


In [ ]:
def tokenize_and_align_labels(examples):
    """
    Токенизация с выравниванием меток.
    
    При токенизации BERT может разбить одно слово на несколько подтокенов.
    Стратегия: первый подтокен получает метку слова, остальные получают -100 (игнорируются при расчёте loss).
    """
    tokenized_inputs = tokenizer(
        examples["tokens"],
        truncation=True,
        max_length=MAX_LENGTH,
        is_split_into_words=True,  # Входные данные уже токенизированы
        padding=False,  # Padding будет добавлен как data collator
    )
    
    labels = []
    for i, label in enumerate(examples["ner_tags"]):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        
        for word_idx in word_ids:
            if word_idx is None:
                # Специальные токены ([CLS], [SEP], [PAD]) получают -100
                label_ids.append(-100)
            elif word_idx != previous_word_idx:
                # Первый подтокен слова - берём метку слова
                label_ids.append(label[word_idx])
            else:
                # Последующие подтокены того же слова - игнорируем
                label_ids.append(-100)
            previous_word_idx = word_idx
        
        labels.append(label_ids)
    
    tokenized_inputs["labels"] = labels
    return tokenized_inputs


In [ ]:
tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,
    desc="Tokenizing dataset",
)

print("Tokenized dataset:")
print(tokenized_dataset)

print("\nПример токенизированных данных:")
example = tokenized_dataset["train"][0]
print(f"Input IDs length: {len(example['input_ids'])}")
print(f"Labels length: {len(example['labels'])}")


Tokenizing dataset:   0%|          | 0/9000 [00:00<?, ? examples/s]

Tokenizing dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenized dataset:
DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 9000
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 1000
    })
})

Пример токенизированных данных:
Input IDs length: 28
Labels length: 28


## 7. Initialize Model


In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id,
)

if DEVICE.type != "cpu":
    model = model.to(DEVICE)

print(f"Model loaded: {MODEL_NAME}")
print(f"Number of parameters: {model.num_parameters():,}")
print(f"Model device: {next(model.parameters()).device}")


Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-multilingual-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model loaded: bert-base-multilingual-cased
Number of parameters: 177,272,845
Model device: cpu


## 8. Metrics


In [ ]:
def compute_metrics(eval_preds):
    """
    Вычисление метрик с использованием seqeval.
    
    seqeval учитывает BIO-схему и корректно считает метрики для NER.
    """
    predictions, labels = eval_preds
    predictions = np.argmax(predictions, axis=2)
    
    true_predictions = []
    true_labels = []
    
    for prediction, label in zip(predictions, labels):
        true_pred = []
        true_lab = []
        
        for p, l in zip(prediction, label):
            if l != -100:  # Игнорируем padding и специальные токены
                true_pred.append(id2label[p])
                true_lab.append(id2label[l])
        
        true_predictions.append(true_pred)
        true_labels.append(true_lab)
    
    return {
        "precision": precision_score(true_labels, true_predictions),
        "recall": recall_score(true_labels, true_predictions),
        "f1": f1_score(true_labels, true_predictions),
    }


## 9. Training Setup


In [ ]:
data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer,
    padding=True,
    return_tensors="pt",
)


In [ ]:
use_cuda = (DEVICE.type == "cuda")
use_mps = (DEVICE.type == "mps")

training_args = TrainingArguments(
    output_dir=str(OUTPUT_DIR),
    
    learning_rate=LEARNING_RATE,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    
    weight_decay=WEIGHT_DECAY,
    warmup_ratio=WARMUP_RATIO,
    optim="adamw_torch",
    
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    
    logging_dir=str(OUTPUT_DIR / "logs"),
    logging_steps=50,
    report_to="none",
    
    use_mps_device=use_mps,
    fp16=use_cuda,
    dataloader_pin_memory=use_cuda,
    no_cuda=(not use_cuda),
    
    seed=SEED,
    save_total_limit=2,
    remove_unused_columns=True,
)

print("Training arguments configured")
print(f"  - Device: {DEVICE}")
print(f"  - Epochs: {NUM_EPOCHS}")
print(f"  - Batch size: {BATCH_SIZE}")
print(f"  - Gradient accumulation: {GRADIENT_ACCUMULATION_STEPS}")
print(f"  - Effective batch size: {BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS}")
print(f"  - Learning rate: {LEARNING_RATE}")
print(f"  - FP16: {use_cuda}")


Training arguments configured
  - Device: cpu
  - Epochs: 3
  - Batch size: 8
  - Gradient accumulation: 2
  - Effective batch size: 16
  - Learning rate: 2e-05
  - FP16: False


/Users/milana/HSE/advanced_python_2025/python_adv_proj/venv/lib/python3.9/site-packages/transformers/training_args.py:1636: FutureWarning: using `no_cuda` is deprecated and will be removed in version 5.0 of 🤗 Transformers. Use `use_cpu` instead
  warnings.warn(


In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
)

print("Trainer initialized")


Trainer initialized


/var/folders/2p/w7_frqdd6fx2bfd7k744c_wc0000gn/T/ipykernel_63154/1322101375.py:2: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


## 10. Train


In [ ]:
import gc
gc.collect()
if torch.backends.mps.is_available():
    torch.mps.empty_cache()

print("Starting training...")
print("=" * 50)

train_result = trainer.train()

print("=" * 50)
print("Training completed!")


Starting training...


Epoch,Training Loss,Validation Loss,Precision,Recall,F1
1,0.002100,0.000435,1.000000,1.000000,1.000000
2,0.000700,0.000242,1.000000,1.000000,1.000000
3,0.000600,0.000212,1.000000,1.000000,1.000000


Training completed!


In [ ]:
print("\nTraining Results:")
print(f"  Total steps: {train_result.global_step}")
print(f"  Training loss: {train_result.training_loss:.4f}")

metrics = train_result.metrics
if "train_runtime" in metrics:
    runtime = metrics["train_runtime"]
    print(f"  Training time: {runtime:.2f}s ({runtime/60:.2f}min)")
    print(f"  Samples/second: {metrics.get('train_samples_per_second', 'N/A')}")



Training Results:
  Total steps: 1689
  Training loss: 0.0764
  Training time: 1773.52s (29.56min)
  Samples/second: 15.224


## 11. Evaluation


In [ ]:
eval_results = trainer.evaluate()

print("\nEvaluation Results:")
print(f"  Loss: {eval_results['eval_loss']:.4f}")
print(f"  Precision: {eval_results['eval_precision']:.4f}")
print(f"  Recall: {eval_results['eval_recall']:.4f}")
print(f"  F1-score: {eval_results['eval_f1']:.4f}")



Evaluation Results:
  Loss: 0.0004
  Precision: 1.0000
  Recall: 1.0000
  F1-score: 1.0000


In [ ]:
def get_detailed_report(trainer, tokenized_dataset):
    """Генерация детального отчёта по классам."""
    predictions, labels, _ = trainer.predict(tokenized_dataset["validation"])
    predictions = np.argmax(predictions, axis=2)
    
    true_predictions = []
    true_labels = []
    
    for prediction, label in zip(predictions, labels):
        true_pred = []
        true_lab = []
        
        for p, l in zip(prediction, label):
            if l != -100:
                true_pred.append(id2label[p])
                true_lab.append(id2label[l])
        
        true_predictions.append(true_pred)
        true_labels.append(true_lab)
    
    return classification_report(true_labels, true_predictions)

print("\nDetailed Classification Report:")
print("=" * 60)
print(get_detailed_report(trainer, tokenized_dataset))



Detailed Classification Report:
              precision    recall  f1-score   support

        DATE       1.00      1.00      1.00      1000
         LOC       1.00      1.00      1.00       490
        TIME       1.00      1.00      1.00      1000
       TITLE       1.00      1.00      1.00       672
         URL       1.00      1.00      1.00       249
        USER       1.00      1.00      1.00       578

   micro avg       1.00      1.00      1.00      3989
   macro avg       1.00      1.00      1.00      3989
weighted avg       1.00      1.00      1.00      3989



## 12. Save Model


In [ ]:
FINAL_MODEL_DIR.mkdir(parents=True, exist_ok=True)

trainer.save_model(str(FINAL_MODEL_DIR))
tokenizer.save_pretrained(str(FINAL_MODEL_DIR))

label_config = {
    "label_list": LABEL_LIST,
    "label2id": label2id,
    "id2label": {str(k): v for k, v in id2label.items()},
}

with open(FINAL_MODEL_DIR / "label_config.json", "w", encoding="utf-8") as f:
    json.dump(label_config, f, ensure_ascii=False, indent=2)

print(f"\nModel saved to: {FINAL_MODEL_DIR.absolute()}")
print(f"Files saved:")
for f in FINAL_MODEL_DIR.iterdir():
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  - {f.name}: {size_mb:.2f} MB")



Model saved to: /Users/milana/HSE/advanced_python_2025/python_adv_proj/model/trained_model
Files saved:
  - model.safetensors: 676.26 MB
  - tokenizer_config.json: 0.00 MB
  - special_tokens_map.json: 0.00 MB
  - config.json: 0.00 MB
  - tokenizer.json: 2.78 MB
  - label_config.json: 0.00 MB
  - training_args.bin: 0.01 MB
  - vocab.txt: 0.95 MB


## 13. Test Inference


In [ ]:
from transformers import pipeline

# Примечание: для MPS используем device=-1 (CPU), так как pipeline может иметь проблемы с MPS
ner_pipeline = pipeline(
    "ner",
    model=str(FINAL_MODEL_DIR),
    tokenizer=str(FINAL_MODEL_DIR),
    aggregation_strategy="simple",  # Объединяет B- и I- теги
    device=0 if torch.cuda.is_available() else -1,
)

print("Pipeline loaded successfully!")


The tokenizer you are loading from 'trained_model' with an incorrect regex pattern: https://huggingface.co/mistralai/Mistral-Small-3.1-24B-Instruct-2503/discussions/84#69121093e8b480e709447d5e. This will lead to incorrect tokenization. You should set the `fix_mistral_regex=True` flag when loading this tokenizer to fix this issue.
Device set to use cpu


Pipeline loaded successfully!


In [ ]:
test_texts = [
    "Встреча завтра в 15:00 в офисе с Иваном",
    "Meeting on Friday at 5pm in the conference room",
    "Созвон в Zoom 25 декабря в 10:30 с командой разработки",
    "Schedule a call with John on 2025-01-15 at 14:00 via https://meet.google.com/abc-defg",
    "Дейлик с командой в 10:00 в офисе",
    "1-1 с руководителем завтра",
    "Очень важный проект с Алексеем завтра в обед",
    "Позвонить Ивану в 15:00"
]

print("\nTest Inference Results:")
print("=" * 60)

for text in test_texts:
    print(f"\nInput: {text}")
    results = ner_pipeline(text)
    print("Entities:")
    for entity in results:
        print(f"  - {entity['entity_group']}: '{entity['word']}' (score: {entity['score']:.3f})")
    print("-" * 40)



Test Inference Results:

Input: Встреча завтра в 15:00 в офисе с Иваном
Entities:
  - TIME: '15 : 00' (score: 0.999)
  - LOC: 'офисе' (score: 0.984)
  - USER: 'Иваном' (score: 0.932)
----------------------------------------

Input: Meeting on Friday at 5pm in the conference room
Entities:
  - TITLE: 'on Friday' (score: 0.718)
  - TIME: '5pm' (score: 0.999)
  - LOC: 'the' (score: 0.896)
  - LOC: 'conference room' (score: 0.986)
----------------------------------------

Input: Созвон в Zoom 25 декабря в 10:30 с командой разработки
Entities:
  - LOC: 'Zoom' (score: 0.998)
  - DATE: '25 декабря' (score: 0.999)
  - TIME: '10 : 30' (score: 1.000)
  - USER: 'командой разработки' (score: 0.997)
----------------------------------------

Input: Schedule a call with John on 2025-01-15 at 14:00 via https://meet.google.com/abc-defg
Entities:
  - USER: 'John' (score: 0.999)
  - DATE: '2025 - 01 - 15' (score: 1.000)
  - TIME: '14 : 00' (score: 1.000)
  - URL: 'https : / / meet. google. com / abc - d

## Summary

### Результаты:
- Модель сохранена в `./trained_model/`
- Чекпоинты сохранены в `./checkpoints/`

---

